In [ ]:
import sys
from pathlib import Path
# Plot internal temp, humidity, external temp, and electricity using Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# So "from pipeline import ..." works when run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [2]:
from pipeline import build_hourly_dataset
hourly_df, home_profile = build_hourly_dataset()

### 2. Run insights and format for LLM
Elec: peak times, tariff recommendation, schedule suggestion. Plus internal temperature and humidity (mean and range) from your hourly data. All merged into one context for the LLM.

In [3]:
# Build insight context: elec + internal temp + humidity (all loaded via config + insight scripts)
from config import ELECTRICITY_CSV, TEMPERATURE_CSV, HUMIDITY_CSV
from data.loaders import load_elec, load_temperature, load_humidity
from insights import (
    peak_usage_times,
    tariff_recommendation,
    schedule_suggestion,
    temperature_summary,
    humidity_summary,
)

elec = load_elec(ELECTRICITY_CSV)
if elec.empty:
    insight_text = "No electricity data available."
else:
    peaks = peak_usage_times(elec)
    tariff = tariff_recommendation(elec)
    sched = schedule_suggestion(elec)
    insight_text = (
        f"Peak usage hours: {peaks.get('peak_hours', [])}; peak days: {peaks.get('peak_days', [])}. "
        f"Tariff: {tariff.get('recommendation', '')} (peak share {tariff.get('peak_share')}). "
        f"Schedule: {sched.get('message', '')} Best hours: {sched.get('best_hours', [])}."
    )

temp_series = load_temperature(TEMPERATURE_CSV)
hum_series = load_humidity(HUMIDITY_CSV)
temp_insight = temperature_summary(temp_series)
hum_insight = humidity_summary(hum_series)
if temp_insight.get("message") and "Not enough" not in temp_insight["message"]:
    insight_text += " " + temp_insight["message"]
if hum_insight.get("message") and "Not enough" not in hum_insight["message"]:
    insight_text += " " + hum_insight["message"]

print(insight_text[:600], "..." if len(insight_text) > 600 else "")

Peak usage hours: [4, 15, 19, 11, 20]; peak days: ['Fri', 'Thu', 'Tue']. Tariff: Usage is spread away from typical peak; standard single-rate may be fine. Still compare tariffs. (peak share 0.24). Schedule: Consider running washing machine, dishwasher, or EV charging in these hours when usage is typically lower. Best hours: [0, 1, 3, 7]. Internal temperature: mean 18.5°C, range 13.4–22.8°C. Internal humidity: mean 46.7%, range 37.1–64.1%. 


### 3. Plot time series (internal temp, humidity, external temp, electricity)

In [ ]:

# Use hourly_df from build_hourly_dataset (cell 2); ensure we have a copy with valid index
df = hourly_df.dropna(how="all").sort_index()

if df.empty:
    print("No hourly data to plot. Run cells 1–2 and ensure data paths in config point to CSVs.")
else:
    # Build subplots: 4 rows, shared x-axis
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        subplot_titles=(
            "Internal temperature (°C)",
            "Internal humidity (%)",
            "External temperature (°C)",
            "Electricity (kWh)",
        ),
        vertical_spacing=0.06,
    )

    x = df.index

    # Internal temperature
    if "internal_temperature_c" in df.columns:
        fig.add_trace(
            go.Scatter(x=x, y=df["internal_temperature_c"], name="Internal temp", line=dict(color="#1f77b4")),
            row=1,
            col=1,
        )

    # Internal humidity
    if "internal_humidity_pct" in df.columns:
        fig.add_trace(
            go.Scatter(x=x, y=df["internal_humidity_pct"], name="Internal humidity", line=dict(color="#2ca02c")),
            row=2,
            col=1,
        )

    # External temperature (from weather; column name from load_weather)
    if "temperature_celsius" in df.columns:
        fig.add_trace(
            go.Scatter(x=x, y=df["temperature_celsius"], name="External temp", line=dict(color="#ff7f0e")),
            row=3,
            col=1,
        )

    # Electricity usage
    if "elec_kwh" in df.columns:
        fig.add_trace(
            go.Scatter(x=x, y=df["elec_kwh"], name="Elec (kWh)", line=dict(color="#d62728")),
            row=4,
            col=1,
        )

    fig.update_layout(height=600, title_text="Household energy and environment", showlegend=False)
    fig.update_yaxes(title_text="°C", row=1, col=1)
    fig.update_yaxes(title_text="%", row=2, col=1)
    fig.update_yaxes(title_text="°C", row=3, col=1)
    fig.update_yaxes(title_text="kWh", row=4, col=1)
    fig.show()

In [4]:
# Load HF model (uses M2 GPU via MPS, or CUDA/CPU)
import torch
from transformers import pipeline

if torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon (M1/M2/M3) GPU
elif torch.cuda.is_available():
    device = 0
else:
    device = -1  # CPU
print(f"Device: {device}" + (" (Apple M-series GPU)" if device == "mps" else (" (NVIDIA GPU)" if device != -1 else " (CPU)")))

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    dtype=torch.float32 if device == -1 else torch.float16,
    device=device,
)
print(f"Model on: {next(pipe.model.parameters()).device}")

Device: mps (Apple M-series GPU)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model on: mps:0


### 4. Interactive Q&A
Run the cell below once. A text box and **Ask** button will appear in the notebook. Type your question and click **Ask**; the reply appears below (no terminal needed).

In [5]:
# Interactive Q&A: ask follow-ups without re-running the model (type 'q' to quit)
import textwrap
from transformers import GenerationConfig

gen_config = GenerationConfig(
    max_new_tokens=150,
    do_sample=True,
    temperature=0.3,
    pad_token_id=pipe.tokenizer.eos_token_id,
)
# Make it explicit that the data is provided in the message (so the model doesn't say it lacks access)
context = (
    "You are an energy advisor. The text below is THIS household's energy data. "
    "Use ONLY this data to answer. The data is provided here — do not say you don't have it.\n\n"
    "--- HOUSEHOLD DATA ---\n"
    f"{insight_text}\n"
    "--- END DATA ---"
)

# Use a text box + button so input appears in the notebook (no terminal)
from ipywidgets import Text, Button, Output, VBox

txt = Text(placeholder="e.g. When should I run my washing machine?", description="Question:", style={"description_width": "80px"}, layout={"width": "500px"})
out_area = Output()
btn = Button(description="Ask")

def on_ask(_):
    q = txt.value.strip()
    if not q:
        return
    with out_area:
        print(f"You: {q}")
        messages = [{"role": "user", "content": f"{context}\n\nQuestion: {q}"}]
        prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        reply = pipe(prompt, generation_config=gen_config)[0]["generated_text"]
        if prompt in reply:
            reply = reply[len(prompt):].strip()
        for sep in ["assistant\n", "Assistant\n", "\n\n"]:
            if sep in reply:
                reply = reply.split(sep)[-1].strip()
        print(textwrap.fill(reply or "(no reply)", width=72), "\n")

btn.on_click(on_ask)
display(VBox([txt, btn, out_area]))

In [6]:
# Summarise insights in 2–3 sentences (use chat format so instruct model replies)
from transformers import GenerationConfig

user_msg = (
    "You are an energy advisor. The text below is this household's energy data. "
    "Use ONLY this data. Do not say you don't have the data — it is below.\n\n"
    "--- HOUSEHOLD DATA ---\n"
    f"{insight_text}\n"
    "--- END DATA ---\n\n"
    "Summarise the key insights in 2-3 sentences and give one clear action."
)
messages = [{"role": "user", "content": user_msg}]
prompt = pipe.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
gen_config = GenerationConfig(
    max_new_tokens=150,
    do_sample=True,
    temperature=0.3,
    pad_token_id=pipe.tokenizer.eos_token_id,
)
out = pipe(prompt, generation_config=gen_config)
summary = out[0]["generated_text"]
# Keep only the assistant reply (after the last turn marker)
if prompt in summary:
    summary = summary[len(prompt) :].strip()
# Fallback: take last line or strip known prefixes
for sep in ["assistant\n", "Assistant\n", "\n\n"]:
    if sep in summary:
        summary = summary.split(sep)[-1].strip()
import textwrap
print(textwrap.fill(summary or "(no output)", width=72))

The household has significant peak usage during weekdays between 4 PM
and 2 AM, with higher rates applied to that time period. Running
appliances like washing machines and dishwashers during off-peak hours
can help manage costs. Additionally, maintaining internal temperatures
at around 18.5°C and humidity levels within the recommended range of
37.1%-64.1% will contribute to more efficient use of energy.
